# Section 6.2 -- AutoGen
## Multi-Agent AI Systems -- Step by Step

**Author:** Suresh D R | AI Product Developer & Technology Mentor

---

AutoGen is Microsoft's framework where agents **talk directly to each other**.

Every agent reads the full conversation before responding -- and replies directly to what others said.

**Domain: EdTech -- Online Learning Platform**

| Step | What You Build |
|---|---|
| 3 | Simple Q&A -- Course recommendation assistant |
| 4 | Curriculum Debate -- two agents debate a course syllabus |
| 5 | Product Launch Decision -- 4 agents debate launching a new AI course |
| 6 | Content Review Pipeline -- Writer, Reviewer, Editor, Publisher |

---
## Step 1 -- Install

In [1]:
# Step 1 — Run this cell first. Then RESTART SESSION before running anything else.
!pip install -q "ag2[openai]==0.14.0"

print("")
print("IMPORTANT: Go to Runtime > Restart Session now.")
print("After restart, run all cells from Step 2 onwards.")
print("Do NOT re-run this install cell after restart.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s eta 0:00:00

IMPORTANT: Go to Runtime > Restart Session now.
After restart, run all cells from Step 2 onwards.
Do NOT re-run this install cell after restart.


---
## Step 2 -- Setup

In [ ]:
import os, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)
os.environ["AUTOGEN_USE_DOCKER"] = "0"

from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

OPENAI_API_KEY = "yr_api_key"

llm_config = {
    "config_list": [{"model": "gpt-3.5-turbo", "api_key": OPENAI_API_KEY}],
    "temperature": 0.7,
    "timeout": 120
}

print("Setup complete!")

Setup complete!


---
## Step 3 -- Simple Conversation: Course Recommendation Assistant

The simplest AutoGen pattern -- one **AssistantAgent** + one **UserProxyAgent**.

The assistant reads the student's message and replies directly.
When done, it says **TERMINATE** to end the conversation.

**Use case:** A student asks which AI course to take next based on their background.

In [4]:
course_advisor = AssistantAgent(
    name="Course_Advisor",
    system_message=(
        "You are a senior course advisor at an online AI and data science learning platform. "
        "Help students choose the right courses based on their background and goals. "
        "The platform offers: Python Basics, Data Analysis, Machine Learning, Deep Learning, "
        "NLP, Computer Vision, MLOps, and Agentic AI. "
        "Give a clear recommendation with reasons. Suggest a learning path. "
        "Keep your answer focused and practical. End with TERMINATE."
    ),
    llm_config=llm_config
)

student = UserProxyAgent(
    name="Student",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    code_execution_config=False,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
)

print("Starting course recommendation conversation...")
print("=" * 55)

student.initiate_chat(
    course_advisor,
    message=(
        "Hi, I am a software developer with 3 years of Python experience. "
        "I know basic statistics and have done one small ML project using scikit-learn. "
        "My goal is to become an AI engineer who can build and deploy real AI products. "
        "Which courses should I take and in what order?"
    )
)

Starting course recommendation conversation...


ChatResult(chat_id=128289220125654700169554594553090182927, chat_history=[{'content': 'Hi, I am a software developer with 3 years of Python experience. I know basic statistics and have done one small ML project using scikit-learn. My goal is to become an AI engineer who can build and deploy real AI products. Which courses should I take and in what order?', 'role': 'assistant', 'name': 'Student'}, {'content': 'Based on your background and goal of becoming an AI engineer, I recommend the following learning path for you:\n\n1. **Machine Learning**: Since you have some experience with scikit-learn, taking a more in-depth Machine Learning course will help you solidify your understanding of different algorithms, model evaluation techniques, and hyperparameter tuning.\n\n2. **Deep Learning**: Deep Learning will introduce you to neural networks, which are essential for building more complex AI models. You will learn about deep learning frameworks like TensorFlow and Keras, enabling you to work

---
## Step 4 -- Curriculum Debate: Two Agents Refine a Syllabus

**Curriculum Designer** proposes a syllabus for a new Agentic AI course.

**Learning Expert** reads it and challenges weak points:
- Is the sequence logical?
- Is the difficulty progression right?
- Are the most important topics covered?

They go back and forth until the Expert says **APPROVED** and **TERMINATE**.

This shows how AutoGen agents respond directly to each other's specific points.

In [5]:
designer = AssistantAgent(
    name="Curriculum_Designer",
    system_message=(
        "You are designing the syllabus for a 6-week Agentic AI course for working professionals. "
        "The students have basic Python and ML knowledge. "
        "Propose a week-by-week syllabus with topic, key concepts, and one hands-on project per week. "
        "When the Learning Expert raises a concern, revise the syllabus to address it. "
        "Keep each response to 4-5 lines. Say TERMINATE when the Expert approves."
    ),
    llm_config=llm_config
)

expert = AssistantAgent(
    name="Learning_Expert",
    system_message=(
        "You are a senior learning design expert reviewing an Agentic AI course syllabus. "
        "Check for: logical topic progression, appropriate difficulty curve, practical projects, "
        "and whether key topics like LangGraph, AutoGen, tool use, and deployment are covered. "
        "Raise one specific concern per response. "
        "When the syllabus is solid and complete, say APPROVED and TERMINATE. "
        "Keep responses to 2-3 sentences."
    ),
    llm_config=llm_config
)

print("Starting curriculum debate...")
print("=" * 55)

expert.initiate_chat(
    designer,
    message=(
        "We need to launch a 6-week Agentic AI course for working professionals. "
        "Students have basic Python and ML knowledge but no experience with agents or LLMs. "
        "Please propose a week-by-week syllabus."
    ),
    max_turns=8
)

Starting curriculum debate...


ChatResult(chat_id=106030035887848817547617313188102464236, chat_history=[{'content': 'We need to launch a 6-week Agentic AI course for working professionals. Students have basic Python and ML knowledge but no experience with agents or LLMs. Please propose a week-by-week syllabus.', 'role': 'assistant', 'name': 'Learning_Expert'}, {'content': 'Week 1: Introduction to Agentic AI\nKey concepts: What is Agentic AI, types of agents, agent environment, rationality\nHands-on project: Implement a simple agent in Python\n\nWeek 2: Intelligent Agents\nKey concepts: Agent perception, reasoning, decision making, and learning\nHands-on project: Develop an intelligent agent for a basic game\n\nWeek 3: Multi-Agent Systems\nKey concepts: MAS architecture, coordination, communication, cooperation, and competition\nHands-on project: Simulate a multi-agent system for traffic management\n\nWeek 4: Reinforcement Learning\nKey concepts: RL basics, Markov decision processes, Q-learning, and policy gradients

---
## Step 5 -- Product Launch Decision: 4-Agent Group Debate

**Scenario:** The EdTech platform is deciding whether to launch a premium Agentic AI Bootcamp
priced at Rs 45,000 targeting working professionals.

Four specialists debate from different angles:

- **Product Head** -- market fit, pricing, positioning
- **Finance Head** -- revenue, costs, breakeven
- **Academic Director** -- content quality, learning outcomes
- **Marketing Head** -- target audience, competition, demand

**GroupChatManager** picks who speaks next based on the conversation.
Every agent reads what the others said before responding.

In [6]:
product = AssistantAgent(
    name="Product_Head",
    system_message=(
        "You are the Product Head of an online EdTech platform. "
        "Evaluate the Agentic AI Bootcamp product: market fit, pricing strategy, unique positioning. "
        "Challenge assumptions about pricing and audience from other agents. "
        "Keep responses to 2-3 sentences. Say TERMINATE when you agree with the final decision."
    ),
    llm_config=llm_config
)

finance = AssistantAgent(
    name="Finance_Head",
    system_message=(
        "You are the Finance Head of an online EdTech platform. "
        "Evaluate the bootcamp: revenue potential, instructor costs, marketing spend, breakeven students. "
        "Be specific with rupee figures and student numbers. Challenge weak financial assumptions. "
        "Keep responses to 2-3 sentences. Say TERMINATE when you agree with the decision."
    ),
    llm_config=llm_config
)

academic = AssistantAgent(
    name="Academic_Director",
    system_message=(
        "You are the Academic Director of an online EdTech platform. "
        "Evaluate the bootcamp for content quality, learning outcomes, and instructor availability. "
        "Flag if the curriculum is too shallow or too advanced for the target audience. "
        "Keep responses to 2-3 sentences. Say TERMINATE when you are satisfied with the plan."
    ),
    llm_config=llm_config
)

marketing = AssistantAgent(
    name="Marketing_Head",
    system_message=(
        "You are the Marketing Head of an online EdTech platform. "
        "Evaluate the bootcamp: target audience size, competitor landscape, demand signals, "
        "and realistic enrollment projections for a Rs 45,000 bootcamp. "
        "Keep responses to 2-3 sentences. Say TERMINATE when consensus is reached."
    ),
    llm_config=llm_config
)

ceo = UserProxyAgent(
    name="CEO",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    default_auto_reply="Continue the debate and reach a clear launch decision.",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
)

group_chat = GroupChat(
    agents=[ceo, product, finance, academic, marketing],
    messages=[],
    max_round=14,
    speaker_selection_method="auto"
)

chat_manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config
)

print("4-agent product launch debate starting...")
print("Agents: Product_Head | Finance_Head | Academic_Director | Marketing_Head")
print("=" * 55)

ceo.initiate_chat(
    chat_manager,
    message=(
        "We are considering launching a premium Agentic AI Bootcamp. "
        "Details: 8-week intensive program, live sessions, projects, mentorship. "
        "Price: Rs 45,000 per student. Target: working professionals wanting to upskill in AI. "
        "We need 40 students minimum to break even. "
        "Competitor platforms (Coursera, Udemy) have similar content at Rs 5,000 to Rs 15,000. "
        "Our differentiation: live mentorship, real projects, job referral network. "
        "Debate the launch decision from your area of expertise. "
        "Should we launch, delay, or change the approach? End with TERMINATE when all agree."
    )
)

4-agent product launch debate starting...
Agents: Product_Head | Finance_Head | Academic_Director | Marketing_Head


ChatResult(chat_id=63681915523686647253432907643643072862, chat_history=[{'content': 'We are considering launching a premium Agentic AI Bootcamp. Details: 8-week intensive program, live sessions, projects, mentorship. Price: Rs 45,000 per student. Target: working professionals wanting to upskill in AI. We need 40 students minimum to break even. Competitor platforms (Coursera, Udemy) have similar content at Rs 5,000 to Rs 15,000. Our differentiation: live mentorship, real projects, job referral network. Debate the launch decision from your area of expertise. Should we launch, delay, or change the approach? End with TERMINATE when all agree.', 'role': 'assistant', 'name': 'CEO'}, {'content': 'The pricing of Rs 45,000 for the Agentic AI Bootcamp seems high compared to competitors like Coursera and Udemy. While the live mentorship, projects, and job referral network are valuable differentiators, the price point might deter potential customers. Consider revisiting the pricing strategy to al

In [7]:
# ============================================
# Pretty-print the agent conversation
# ============================================

def show_conversation(messages, title="AGENT CONVERSATION"):
    print("=" * 70)
    print(f"  {title}")
    print("=" * 70)
    for i, msg in enumerate(messages, 1):
        speaker = msg.get("name", msg.get("role", "Unknown"))
        content = (msg.get("content") or "").strip()
        if not content:
            continue
        print()
        print(f"[{i}] {speaker}")
        print("-" * 70)
        print(content)
    print()
    print("=" * 70)
    print(f"  END OF CONVERSATION ({len(messages)} messages)")
    print("=" * 70)

# The group chat keeps the full history after the run:
show_conversation(group_chat.messages, title="PRODUCT LAUNCH DEBATE - 4 AGENTS")

  PRODUCT LAUNCH DEBATE - 4 AGENTS

[1] CEO
----------------------------------------------------------------------
We are considering launching a premium Agentic AI Bootcamp. Details: 8-week intensive program, live sessions, projects, mentorship. Price: Rs 45,000 per student. Target: working professionals wanting to upskill in AI. We need 40 students minimum to break even. Competitor platforms (Coursera, Udemy) have similar content at Rs 5,000 to Rs 15,000. Our differentiation: live mentorship, real projects, job referral network. Debate the launch decision from your area of expertise. Should we launch, delay, or change the approach? End with TERMINATE when all agree.

[2] Product_Head
----------------------------------------------------------------------
The pricing of Rs 45,000 for the Agentic AI Bootcamp seems high compared to competitors like Coursera and Udemy. While the live mentorship, projects, and job referral network are valuable differentiators, the price point might deter p

---
## Step 6 -- Content Review Pipeline

**Scenario:** A new course description needs to pass through multiple reviewers before going live.

- **Content Writer** drafts the course description
- **Accuracy Reviewer** checks that all claims about the course content are correct
- **Marketing Reviewer** checks that the tone is engaging and the value proposition is clear
- **Publisher** approves only when both reviewers say OK

Each reviewer reads the full conversation -- they see what earlier reviewers said
and can build on those points before responding.

In [8]:
content_writer = AssistantAgent(
    name="Content_Writer",
    system_message=(
        "You write course descriptions for an AI learning platform. "
        "Write a short course description (4-5 sentences) for an Agentic AI course. "
        "Include: what students will learn, who it is for, key tools covered (LangGraph, AutoGen, CrewAI), "
        "and the outcome after completing the course. "
        "When reviewers ask for changes, update the description and re-submit. "
        "When Publisher approves, say TERMINATE."
    ),
    llm_config=llm_config
)

accuracy_reviewer = AssistantAgent(
    name="Accuracy_Reviewer",
    system_message=(
        "You review course descriptions for technical accuracy. "
        "Check that: tool names are correct, claims about what students learn are realistic, "
        "and no misleading statements are made about outcomes. "
        "If something is inaccurate or misleading, ask Writer to fix one specific thing. "
        "When technically accurate, say ACCURACY OK. Keep responses to 1-2 sentences."
    ),
    llm_config=llm_config
)

marketing_reviewer = AssistantAgent(
    name="Marketing_Reviewer",
    system_message=(
        "You review course descriptions for marketing effectiveness. "
        "Check that: the value proposition is clear, the target audience is defined, "
        "and the tone is engaging and motivating for working professionals. "
        "If the marketing is weak, ask Writer to improve one specific thing. "
        "When the description is compelling and clear, say MARKETING OK. Keep responses to 1-2 sentences."
    ),
    llm_config=llm_config
)

publisher = AssistantAgent(
    name="Publisher",
    system_message=(
        "You give final approval to publish course descriptions. "
        "Only approve when Accuracy_Reviewer said ACCURACY OK and Marketing_Reviewer said MARKETING OK. "
        "If both have approved, say PUBLISHED and TERMINATE. "
        "If not yet approved by both, clearly state what is still needed. "
        "Keep responses to 1-2 sentences."
    ),
    llm_config=llm_config
)

platform = UserProxyAgent(
    name="Platform",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    default_auto_reply="Continue the review process.",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
)

review_chat = GroupChat(
    agents=[platform, content_writer, accuracy_reviewer, marketing_reviewer, publisher],
    messages=[],
    max_round=12,
    speaker_selection_method="auto"
)

review_manager = GroupChatManager(
    groupchat=review_chat,
    llm_config=llm_config
)

print("Content review pipeline starting...")
print("Agents: Content_Writer | Accuracy_Reviewer | Marketing_Reviewer | Publisher")
print("=" * 55)

platform.initiate_chat(
    review_manager,
    message=(
        "We need a course description for our new Agentic AI course. "
        "The course covers: what agents are, LangGraph workflows, AutoGen debates, "
        "CrewAI teams, tool calling, human-in-the-loop, and building real agent projects. "
        "Target audience: software developers and data scientists with basic Python and ML knowledge. "
        "Please write the description, then Accuracy_Reviewer and Marketing_Reviewer must review it. "
        "Publisher approves only after both reviewers are satisfied."
    )
)

Content review pipeline starting...
Agents: Content_Writer | Accuracy_Reviewer | Marketing_Reviewer | Publisher


ChatResult(chat_id=196366718069880282711682963537280560422, chat_history=[{'content': 'We need a course description for our new Agentic AI course. The course covers: what agents are, LangGraph workflows, AutoGen debates, CrewAI teams, tool calling, human-in-the-loop, and building real agent projects. Target audience: software developers and data scientists with basic Python and ML knowledge. Please write the description, then Accuracy_Reviewer and Marketing_Reviewer must review it. Publisher approves only after both reviewers are satisfied.', 'role': 'assistant', 'name': 'Platform'}, {'content': 'In the Agentic AI course, students will explore the concept of agents and delve into LangGraph workflows, AutoGen debates, and CrewAI teams. This course is designed for software developers and data scientists with basic Python and machine learning knowledge. Key tools covered include LangGraph, AutoGen, and CrewAI. By the end of the course, students will be equipped to implement tool calling, 

---
## Summary

| Step | Pattern | Agents | Use Case |
|---|---|---|---|
| 3 | Simple conversation | 1 advisor + 1 student | Course recommendation |
| 4 | Two-agent debate | Designer vs Expert | Curriculum syllabus refinement |
| 5 | Group debate | 4 specialists + CEO | Bootcamp launch decision |
| 6 | Review pipeline | Writer + 3 reviewers | Course description approval |

---

### Key AutoGen Concepts

**AssistantAgent** -- the AI agent. The `system_message` defines its role and how it challenges others.

**UserProxyAgent** -- starts the conversation. `is_termination_msg` stops when agent says TERMINATE.

**GroupChat** -- holds all agents. `max_round` prevents infinite loops.

**GroupChatManager** -- reads the conversation, picks who speaks next.

**The AutoGen rule:** Every agent reads the **full conversation** before responding.
They respond to what was **said** -- not to a shared state dictionary.

**Next: Section 6.3 -- CrewAI**